In [ ]:
""" 
Intraday RSI Divergence Scanner (V2)
- Timeframes: 3min (RSI7) + 30min (RSI14)
- Scan frequency: Every N minutes
- Trading hours: 06:30 - 13:05 Pacific Time
- Timezone: America/Los_Angeles (Pacific Time)
- Includes: Hidden Divergence detection
"""

SCAN_INTERVAL_MINUTES = 1

import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg, _import_name in [
    ("schedule",           "schedule"),
    ("pandas",             "pandas"),
    ("pandas_ta",          "pandas_ta"),
    ("polygon-api-client", "polygon"),
    ("pytz",               "pytz"),
    ("requests",           "requests"),
]:
    try:
        __import__(_import_name)
    except ImportError:
        print(f"Installing {_pkg} ...")
        _pip(_pkg)

import time
import datetime
import threading
import schedule
import pandas as pd
import pandas_ta as ta
import pytz
import requests
from polygon import RESTClient
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================= Configuration =================
API_KEY = "Your API Key"
client  = RESTClient(API_KEY)

TICKERS = ["QQQ", "SPY", "SOXS", "UVXY", "SOXX",
           "ONDS","TSLA"]

TIMEFRAMES_MAP = {
    "3min":  {"multiplier": 3,  "timespan": "minute", "bars_limit": 200, "rsi_length": 7},   # RSI 7
    "30min": {"multiplier": 30, "timespan": "minute", "bars_limit": 120, "rsi_length": 14},  # RSI 14
}

PIVOT_WINDOW = 3
TARGET_TZ    = "America/Los_Angeles"
PACIFIC      = pytz.timezone(TARGET_TZ)
MARKET_OPEN  = datetime.time(6, 30)
MARKET_CLOSE = datetime.time(13, 5)

RETRY_TIMES   = 2
RETRY_WAIT    = 2
MAX_WORKERS   = 8

MARKET_ETFS         = ["QQQ", "SPY"]
SOXS_ETF            = "SOXS"
INVERSE_ETF         = "UVXY"
SIGNAL_ONLY_TICKERS = {"NVDA,TSLA,ONDS"}

LOG_FILE = "scan_log.txt"

# ================= Telegram Configuration =================
TELEGRAM_TOKEN   = "Your Telegram Bot Token"
TELEGRAM_CHAT_ID = "Your Telegram Chat ID"

_scan_count = {"n": 0}

def send_telegram(msg: str):
    try:
        url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
        requests.post(url, json={
            "chat_id": TELEGRAM_CHAT_ID,
            "text": msg,
            "parse_mode": "HTML"
        }, timeout=10)
    except Exception as e:
        log(f"  [Telegram send failed] {e}")

# ================= Logging =================
_log_handle = None
_log_lock   = threading.Lock()

def _get_log():
    global _log_handle
    if _log_handle is None or _log_handle.closed:
        _log_handle = open(LOG_FILE, "a", encoding="utf-8")
    return _log_handle

def log(msg=""):
    print(msg)
    with _log_lock:
        f = _get_log()
        f.write(msg + "\n")
        f.flush()

# ================= Data Retrieval =================
def get_data(ticker, multiplier, timespan, bars_limit):
    now   = datetime.datetime.now(PACIFIC)
    start = now - datetime.timedelta(days=10)

    for attempt in range(1, RETRY_TIMES + 1):
        try:
            aggs = list(client.list_aggs(
                ticker, multiplier, timespan,
                start.strftime("%Y-%m-%d"),
                now.strftime("%Y-%m-%d"),
                limit=50000
            ))
            if not aggs:
                return None

            df = pd.DataFrame(aggs)
            df["timestamp"] = (
                pd.to_datetime(df["timestamp"], unit="ms")
                .dt.tz_localize("UTC")
                .dt.tz_convert(TARGET_TZ)
            )
            df = (df.set_index("timestamp")
                    .sort_index()
                    .rename(columns={"open": "Open", "high": "High",
                                     "low": "Low", "close": "Close",
                                     "volume": "Volume"})
                    .between_time("06:30", "13:00"))

            return df.tail(bars_limit) if not df.empty else None

        except Exception as e:
            err = str(e)
            if "429" in err or "too many" in err.lower():
                if attempt < RETRY_TIMES:
                    time.sleep(RETRY_WAIT)
                else:
                    log(f"  [Rate limited 429] {ticker} {multiplier}{timespan} — retried {RETRY_TIMES} times, skipping")
            else:
                log(f"  [Data error] {ticker} {multiplier}{timespan}: {e}")
                return None

    return None

# ================= Single Task: Fetch Data + Compute Signals =================
def fetch_and_scan(ticker, tf_label, cfg):
    results = []
    df = get_data(ticker, cfg["multiplier"], cfg["timespan"], cfg["bars_limit"])
    if df is None or len(df) < 50:
        return results
    
    rsi_length = cfg.get("rsi_length", 14)  # Get RSI length from config
    
    try:
        rsi = ta.rsi(df["Close"], length=rsi_length)
        if rsi is None:
            return results
        df["RSI"] = rsi
    except Exception:
        return results

    for bar_mode, bar_idx in [("0-Bar", -1), ("1-Bar", -2)]:
        if bar_idx == -2 and len(df) <= 5:
            continue
        try:
            r = check_divergence(df, bar_idx)
            if r:
                bar_dt = df.index[bar_idx]
                results.append({
                    "dt":      bar_dt,
                    "ticker":  ticker,
                    "tf":      tf_label,
                    "mode":    bar_mode,
                    "time":    bar_dt.strftime("%m-%d %H:%M"),
                    "type":    r["type"],
                    "ref":     r["ref"],
                    "refdate": r["refdate"],
                    "detail":  r["detail"],
                    "bullish": r["bullish"],
                    "hidden":  r.get("hidden", False),
                })
        except Exception:
            pass

    return results

# ================= Divergence Detection (Including Hidden Divergence) =================
def check_divergence(df, target_idx):
    """
    Detects four types of divergence:
    
    Regular Divergence - Trend reversal signals:
    - Bullish divergence: Price makes new low + RSI does not make new low -> Bullish
    - Bearish divergence: Price makes new high + RSI does not make new high -> Bearish
    
    Hidden Divergence - Trend continuation signals:
    - Hidden bullish divergence: Price makes higher low + RSI makes new low -> Uptrend continuation
    - Hidden bearish divergence: Price makes lower high + RSI makes new high -> Downtrend continuation
    """
    if "RSI" not in df.columns:
        return None

    abs_idx = len(df) + target_idx if target_idx < 0 else target_idx
    if abs_idx < PIVOT_WINDOW + 1:
        return None

    curr_low  = df["Low"].iloc[abs_idx]
    curr_high = df["High"].iloc[abs_idx]
    curr_rsi  = df["RSI"].iloc[abs_idx]
    if pd.isna(curr_rsi):
        return None

    # Find pivot points
    pivot_lows, pivot_highs = [], []
    for i in range(PIVOT_WINDOW, abs_idx):
        if i + PIVOT_WINDOW >= abs_idx:
            break
        
        # Pivot Low
        v = df["Low"].iloc[i]
        if (v < df["Low"].iloc[i - PIVOT_WINDOW:i].min() and
                v < df["Low"].iloc[i + 1:i + PIVOT_WINDOW + 1].min() and
                not pd.isna(df["RSI"].iloc[i])):
            pivot_lows.append(i)
        
        # Pivot High
        v = df["High"].iloc[i]
        if (v > df["High"].iloc[i - PIVOT_WINDOW:i].max() and
                v > df["High"].iloc[i + 1:i + PIVOT_WINDOW + 1].max() and
                not pd.isna(df["RSI"].iloc[i])):
            pivot_highs.append(i)

    # ==================== Regular Bullish Divergence ====================
    # Price makes new low + RSI does not make new low -> Bullish reversal
    for rank, p in enumerate(reversed(pivot_lows[-3:])):
        h_low, h_rsi, h_date = df["Low"].iloc[p], df["RSI"].iloc[p], df.index[p]
        if curr_low < h_low and curr_rsi > h_rsi and curr_rsi < 60 and (curr_rsi - h_rsi) > 1.0:
            return {
                "type":    "Bullish Div",
                "ref":     f"Pivot-{rank + 1}",
                "refdate": h_date.strftime("%m-%d %H:%M"),
                "detail":  f"P:{h_low:.2f}->{curr_low:.2f}  RSI:{h_rsi:.0f}->{curr_rsi:.0f}",
                "bullish": True,
                "hidden":  False,
            }

    # ==================== Regular Bearish Divergence ====================
    # Price makes new high + RSI does not make new high -> Bearish reversal
    for rank, p in enumerate(reversed(pivot_highs[-3:])):
        h_high, h_rsi, h_date = df["High"].iloc[p], df["RSI"].iloc[p], df.index[p]
        if curr_high > h_high and curr_rsi < h_rsi and curr_rsi > 40 and (h_rsi - curr_rsi) > 1.0:
            return {
                "type":    "Bearish Div",
                "ref":     f"Pivot-{rank + 1}",
                "refdate": h_date.strftime("%m-%d %H:%M"),
                "detail":  f"P:{h_high:.2f}->{curr_high:.2f}  RSI:{h_rsi:.0f}->{curr_rsi:.0f}",
                "bullish": False,
                "hidden":  False,
            }

    # ==================== Hidden Bullish Divergence (Continuation) ====================
    # Price makes higher low + RSI makes new low -> Uptrend continuation
    for rank, p in enumerate(reversed(pivot_lows[-3:])):
        h_low, h_rsi, h_date = df["Low"].iloc[p], df["RSI"].iloc[p], df.index[p]
        # Price: current low > historical low (higher low)
        # RSI: current RSI < historical RSI (RSI new low)
        if curr_low > h_low and curr_rsi < h_rsi and curr_rsi < 55 and (h_rsi - curr_rsi) > 1.0:
            return {
                "type":    "H.Bullish Div",  # H = Hidden
                "ref":     f"Pivot-{rank + 1}",
                "refdate": h_date.strftime("%m-%d %H:%M"),
                "detail":  f"P:{h_low:.2f}->{curr_low:.2f}(HL)  RSI:{h_rsi:.0f}->{curr_rsi:.0f}(LL)",
                "bullish": True,
                "hidden":  True,
            }

    # ==================== Hidden Bearish Divergence (Continuation) ====================
    # Price makes lower high + RSI makes new high -> Downtrend continuation
    for rank, p in enumerate(reversed(pivot_highs[-3:])):
        h_high, h_rsi, h_date = df["High"].iloc[p], df["RSI"].iloc[p], df.index[p]
        # Price: current high < historical high (lower high)
        # RSI: current RSI > historical RSI (RSI new high)
        if curr_high < h_high and curr_rsi > h_rsi and curr_rsi > 45 and (curr_rsi - h_rsi) > 1.0:
            return {
                "type":    "H.Bearish Div",  # H = Hidden
                "ref":     f"Pivot-{rank + 1}",
                "refdate": h_date.strftime("%m-%d %H:%M"),
                "detail":  f"P:{h_high:.2f}->{curr_high:.2f}(LH)  RSI:{h_rsi:.0f}->{curr_rsi:.0f}(HH)",
                "bullish": False,
                "hidden":  True,
            }

    return None

# ================= Summary & Telegram Alerts =================
def print_summary(raw, now_pt, W):
    log(f"\n{'─' * W}")
    log(f"  📊 Market Summary  ({now_pt.strftime('%H:%M')} PT)")
    log(f"{'─' * W}")

    grouped = {}
    for row in raw:
        grouped.setdefault((row["ticker"], row["tf"]), []).append(row)

    deduped = {}
    for key, rows in grouped.items():
        r0 = next((r for r in rows if r["mode"] == "0-Bar"), None)
        r1 = next((r for r in rows if r["mode"] == "1-Bar"), None)
        out = []
        if r0 and r1 and r0["dt"] == r1["dt"]:
            merged = dict(r0); merged["mode"] = "Confirmed"; out.append(merged)
        else:
            if r0: out.append(r0)
            if r1: out.append(r1)
        if out:
            deduped[key] = out

    tg_lines    = []
    printed_any = False

    # Summary collection: 3min and 30min separated, includes ETF + UVXY
    summary = {
        "3min":  {"0-Bar": [], "1-Bar": [], "Confirmed": []},
        "30min": {"1-Bar": []},
    }

    for tf in ["3min", "30min"]:
        for mode in ["Confirmed", "0-Bar", "1-Bar"]:
            # Skip 30min 0-Bar and Confirmed (only need 1-Bar)
            if tf == "30min" and mode != "1-Bar":
                continue

            uvxy_rows = deduped.get((INVERSE_ETF, tf), [])
            uvxy_row  = next((r for r in uvxy_rows if r["mode"] == mode), None)

            soxs_rows = deduped.get((SOXS_ETF, tf), [])
            soxs_row  = next((r for r in soxs_rows if r["mode"] == mode), None)
            soxs_str  = ""
            if soxs_row:
                hidden_tag = "H" if soxs_row.get("hidden") else ""
                soxs_lbl  = f"{hidden_tag}Bearish Div" if not soxs_row["bullish"] else f"{hidden_tag}Bullish Div"
                soxs_dir  = "Short weakening" if not soxs_row["bullish"] else "Short strengthening"
                soxs_time = soxs_row["time"]
                soxs_str  = f"  SOXS {soxs_lbl}({soxs_dir}) @{soxs_time}"

            # UVXY summary
            if uvxy_row:
                hidden_tag = "H" if uvxy_row.get("hidden") else ""
                uvxy_short = f"UVXY{hidden_tag}Bearish" if not uvxy_row["bullish"] else f"UVXY{hidden_tag}Bullish Div"
                uvxy_label = f"{hidden_tag}Bearish Div (fear fading)" if not uvxy_row["bullish"] else f"{hidden_tag}Bullish Div (fear rising)"
                uvxy_time  = uvxy_row["time"]
                uvxy_bull_mkt = not uvxy_row["bullish"]
                summary[tf][mode].append(uvxy_short)
            else:
                uvxy_label    = None
                uvxy_time     = None
                uvxy_bull_mkt = None

            for etf in MARKET_ETFS:
                etf_rows = deduped.get((etf, tf), [])
                etf_row  = next((r for r in etf_rows if r["mode"] == mode), None)

                # ETF summary (collect regardless of UVXY presence)
                if etf_row:
                    etf_bull      = etf_row["bullish"]
                    hidden_tag    = "H" if etf_row.get("hidden") else ""
                    etf_short     = f"{etf}{hidden_tag}{'📈' if etf_bull else '📉'}"
                    summary[tf][mode].append(etf_short)

                # Confluence check (requires UVXY)
                if etf_row and uvxy_row:
                    etf_bull   = etf_row["bullish"]
                    hidden_tag = "H" if etf_row.get("hidden") else ""
                    etf_label  = f"{hidden_tag}Bullish Div" if etf_bull else f"{hidden_tag}Bearish Div"
                    mode_tag   = f"[{mode}]"
                    etf_time   = etf_row["time"]

                    if etf_bull and uvxy_bull_mkt:
                        verdict = "📈 Bullish confirmed — confluence"
                    elif not etf_bull and not uvxy_bull_mkt:
                        verdict = "📉 Bearish confirmed — confluence"
                    elif etf_bull and not uvxy_bull_mkt:
                        verdict = "⚠️ Conflicting signals (ETF bullish div but UVXY fear rising)"
                    else:
                        verdict = "⚠️ Conflicting signals (ETF bearish div but UVXY fear fading)"

                    soxs_part = f"  |{soxs_str}" if soxs_str else ""
                    log(f"  {mode_tag:<8} {tf:<6} {etf:<6} "
                        f"{etf_label:<12} @{etf_time}  |  "
                        f"UVXY {uvxy_label:<22} @{uvxy_time}  ->  {verdict}{soxs_part}")
                    tg_lines.append(f"{mode_tag} {tf} {etf} {etf_label} @{etf_time}\nUVXY {uvxy_label} @{uvxy_time}\n-> {verdict}")
                    printed_any = True

    if not printed_any:
        log("  -- No QQQ/SPY/SOXS/UVXY confluence signals this round --")

    # Individual stock signals
    signal_lines = []
    for ticker in sorted(SIGNAL_ONLY_TICKERS):
        for tf in ["3min", "30min"]:
            rows = deduped.get((ticker, tf), [])
            for row in rows:
                hidden_tag = "[H]" if row.get("hidden") else ""
                signal_lines.append(
                    f"  [{row['mode']}] {ticker:<6} {tf:<6} {row['time']}  {hidden_tag}{row['type']}  {row['detail']}"
                )

    if signal_lines:
        log(f"\n  {'─' * 50}")
        log(f"  📌 Individual Stock Signals")
        log(f"  {'─' * 50}")
        for l in signal_lines:
            log(l)

    log()

    # -- Send Telegram every 3 scans --
    _scan_count["n"] += 1
    if _scan_count["n"] == 1 or _scan_count["n"] % 3 == 0:
        now_str = now_pt.strftime("%H:%M PT")

        def fmt_line(items, label):
            return f"{label}: {' | '.join(items)}" if items else f"{label}: No signals"

        line_0bar_3min  = fmt_line(summary["3min"]["0-Bar"], "0 Bar-3min(RSI7)")
        line_1bar_3min  = fmt_line(summary["3min"]["1-Bar"], "1 Bar-3min(RSI7)")
        line_1bar_30min = fmt_line(summary["30min"]["1-Bar"], "1 Bar-30min(RSI14)")

        header = f"📊 <b>{now_str}</b>\n{line_0bar_3min}\n{line_1bar_3min}\n{line_1bar_30min}"

        if summary["3min"]["Confirmed"]:
            line_confirm = fmt_line(summary["3min"]["Confirmed"], "Confirmed-3min(RSI7)")
            header += f"\n{line_confirm}"

        if tg_lines:
            tg_msg = header + "\n\n" + "\n\n".join(tg_lines)
        else:
            tg_msg = header + "\n\n-- No confluence signals --"

        send_telegram(tg_msg)

# ================= Single Scan =================
def run_scan():
    now_pt = datetime.datetime.now(PACIFIC)

    if not (MARKET_OPEN <= now_pt.time() <= MARKET_CLOSE):
        log(f"[{now_pt.strftime('%H:%M:%S')} PT] Outside trading hours, skipping.")
        return

    W = 130
    log("\n" + "🟧" * 65)
    log(f"{'=' * W}")
    log(f"  📡  {now_pt.strftime('%Y-%m-%d %H:%M:%S %Z')}  |  3min(RSI7) & 30min(RSI14)  |  {', '.join(TICKERS)}")
    log(f"{'=' * W}")
    log(f"{'Ticker':<8} {'TF':<7} {'Mode':<7} {'Bar Time(PT)':<16} {'Signal':<16} {'Ref':<10} {'Pivot Date':<14} Detail")
    log(f"{'-' * W}")

    raw = []
    tasks = [(ticker, tf_label, cfg)
             for ticker in TICKERS
             for tf_label, cfg in TIMEFRAMES_MAP.items()]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_and_scan, t, tf, cfg): (t, tf)
                   for t, tf, cfg in tasks}
        for future in as_completed(futures):
            try:
                raw.extend(future.result())
            except Exception:
                pass

    display_rows = []
    grouped = {}
    for row in raw:
        grouped.setdefault((row["ticker"], row["tf"]), []).append(row)

    for key, rows in grouped.items():
        r0 = next((r for r in rows if r["mode"] == "0-Bar"), None)
        r1 = next((r for r in rows if r["mode"] == "1-Bar"), None)
        if r0 and r1 and r0["dt"] == r1["dt"]:
            merged = dict(r0); merged["mode"] = "Confirmed"
            display_rows.append(merged)
        else:
            if r0: display_rows.append(r0)
            if r1: display_rows.append(r1)

    display_rows.sort(key=lambda x: (x["dt"], x["mode"]), reverse=True)

    if not display_rows:
        log("  -- No divergence signals found this round --")
    else:
        for row in display_rows:
            hidden_tag = "[H]" if row.get("hidden") else ""
            log(f"{row['ticker']:<8} {row['tf']:<7} {row['mode']:<7} {row['time']:<16} "
                f"{hidden_tag}{row['type']:<16} {row['ref']:<10} {row['refdate']:<14} {row['detail']}")

    print_summary(raw, now_pt, W)

    next_t = (now_pt + datetime.timedelta(minutes=SCAN_INTERVAL_MINUTES)).strftime("%H:%M")
    log(f"  ⏰ Next scan: {next_t} PT")

# ================= Main =================
def main():
    log("=" * 70)
    log("  Intraday RSI Divergence Scanner V2 Started")
    log(f"  Tickers : {', '.join(TICKERS)}")
    log("  Timeframes : 3min (RSI 7) / 30min (RSI 14)")
    log("  Divergence : Regular + Hidden")
    log(f"  Frequency : Every {SCAN_INTERVAL_MINUTES} minute(s)")
    log("  Hours : 06:30 - 13:05 PT")
    log(f"  Log file : {LOG_FILE}")
    log("  Telegram : Sends on 1st scan, then every 3rd scan")
    log("  Exit : Ctrl+C")
    log("=" * 70)
    log("")
    log("  Divergence Guide:")
    log("  ┌─────────────────────────────────────────────────────────────────┐")
    log("  │ Regular Bullish Div : Price new low + RSI no new low -> Reversal│")
    log("  │ Regular Bearish Div : Price new high + RSI no new high-> Reversal│")
    log("  │ Hidden  Bullish Div : Higher low + RSI new low -> Continuation  │")
    log("  │ Hidden  Bearish Div : Lower high + RSI new high -> Continuation │")
    log("  └─────────────────────────────────────────────────────────────────┘")
    log("")

    run_scan()
    schedule.every(SCAN_INTERVAL_MINUTES).minutes.do(run_scan)

    in_jupyter = False
    try:
        get_ipython()  # noqa: F821
        in_jupyter = True
    except NameError:
        pass

    if in_jupyter:
        def _bg():
            while True:
                schedule.run_pending()
                time.sleep(30)
        t = threading.Thread(target=_bg, daemon=True)
        t.start()
        print(f"\n  ✅ Jupyter mode: Background thread running. Full log at {LOG_FILE}")
    else:
        while True:
            schedule.run_pending()
            time.sleep(30)

if __name__ == "__main__":
    main()

  盘中 RSI 背离扫描器 V2 启动
  标的 : QQQ, SPY, SOXS, UVXY, SOXX, ONDS, IREN, TSLA
  周期 : 3min (RSI 7) / 30min (RSI 14)
  背离 : Regular + Hidden (隐藏背离)
  频率 : 每 1 分钟
  时段 : 06:30 - 13:05 PT
  日志 : scan_log.txt
  Telegram : 第1次及之后每3次推送一次
  退出 : Ctrl+C

  背离说明:
  ┌─────────────────────────────────────────────────────────────────┐
  │ Regular 底背离 🟢 : 价格新低 + RSI未新低 → 看涨反转             │
  │ Regular 顶背离 🔴 : 价格新高 + RSI未新高 → 看跌反转             │
  │ Hidden  底背离 🟢 : 价格更高低点 + RSI新低 → 上涨趋势延续       │
  │ Hidden  顶背离 🔴 : 价格更低高点 + RSI新高 → 下跌趋势延续       │
  └─────────────────────────────────────────────────────────────────┘


🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧
  📡  2026-02-20 06:59:12 PST  |  3min(RSI7) & 30min(RSI14)  |  QQQ, SPY, SOXS, UVXY, SOXX, ONDS, IREN, TSLA
标的       周期      模式      K线时间(PT)         信号               对比         Pivot日期        详情
----------------------------------------------------------------------------------------------------------------------------------
UV


🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧🟧
  📡  2026-02-20 07:00:16 PST  |  3min(RSI7) & 30min(RSI14)  |  QQQ, SPY, SOXS, UVXY, SOXX, ONDS, IREN, TSLA
标的       周期      模式      K线时间(PT)         信号               对比         Pivot日期        详情
----------------------------------------------------------------------------------------------------------------------------------
SPY      3min    0-Bar   02-20 06:57      顶背离 🔴            Pivot-2    02-19 13:00    P:684.75→685.04  RSI:83→64
SOXX     3min    0-Bar   02-20 06:57      顶背离 🔴            Pivot-2    02-19 12:03    P:354.46→357.38  RSI:66→59
IREN     3min    0-Bar   02-20 06:57      顶背离 🔴            Pivot-2    02-19 12:03    P:43.12→43.34  RSI:67→64
ONDS     3min    0-Bar   02-20 06:57      顶背离 🔴            Pivot-3    02-19 12:03    P:11.15→11.21  RSI:61→51
UVXY     3min    0-Bar   02-20 06:57      底背离 🟢            Pivot-2    02-19 13:00    P:39.94→39.85  RSI:25→36
SPY      3min    1-Bar   02-20 06:54      顶背离 🔴     